# CSC 2301 · Week 11 — Modules: reusable code in a file

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/csc2301/notebooks/CSC2301_W11_modules_geotools.ipynb)

**IUGB · BSc Mining / Petroleum Engineering** · Instructor: Laurent Kouadio

Since week 3 you have rewritten `block_tonnes`, `gold_ounces` and `vshale` in several notebooks. Engineers
put the functions they trust in a **module**, a `.py` file, and **import** it wherever they need it:
one place to fix a bug, one place to test. This week you build `geotools.py`, your own small toolbox.

**What you will practise:** `import`, `from … import`, `as` · the standard library (`math`, `statistics`,
`collections`) · writing a module with `%%writefile` · docstrings and `help` · `importlib.reload` ·
scripts and `if __name__ == "__main__":`.

Run each cell with **Shift + Enter**, from top to bottom, starting with the two setup cells.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/csc2301/'
FILES = [('data/block_model_pit1.csv', 'block_model_pit1.csv'), ('tests/w11/q1.py', 'tests/q1.py'), ('tests/w11/q2.py', 'tests/q2.py'), ('tests/w11/q3.py', 'tests/q3.py'), ('tests/w11/q4.py', 'tests/q4.py'), ('tests/w11/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

## 1. Three ways to import

In [ ]:
import math                      # the module: use math.sqrt
from statistics import mean      # one name from a module
import numpy as np               # a module with a short alias

print(math.sqrt(2), math.pi)
print(mean([2.1, 2.5, 2.3]))
print(np.array([1, 2, 3]) * 2)

In [ ]:
help(math.hypot)                 # every good function has a docstring
print([n for n in dir(math) if not n.startswith("_")][:12])

## 2. Your own module

A module is just a `.py` file. In a notebook, the magic `%%writefile name.py` **must be the first line** of
the cell: it saves the rest of the cell as a file instead of running it.

In [ ]:
%%writefile units.py
"""Unit conversions used in mining and petroleum engineering."""

FT_PER_M = 3.28084

def m_to_ft(m):
    """Convert metres to feet."""
    return m * FT_PER_M

def psi_to_mpa(psi):
    """Convert pounds per square inch to megapascals."""
    return psi * 0.00689476

In [ ]:
import units

print(units.m_to_ft(1500))
print(units.psi_to_mpa(3000))
help(units)

**Changed the file?** Python imports a module only once per session. After editing, reload it:
`import importlib; importlib.reload(units)` (or restart the runtime).

## 3. Scripts and `__name__`

When a file is **run** (`python report.py`, or `%run report.py` in a notebook), its `__name__` is
`"__main__"`. When it is **imported**, `__name__` is the module name. So code under
`if __name__ == "__main__":` runs only when the file is used as a script.

---
## Exercises

### Q1. Start `geotools.py`

Complete the cell and run it to write `geotools.py`. It must contain the constant `GOLD_OZ_G = 31.1035` and
your week 3 functions `block_tonnes` and `gold_ounces` (which must use `GOLD_OZ_G`), each with a docstring.

In [ ]:
%%writefile geotools.py
"""geotools: small helpers for mining and petroleum calculations."""

GOLD_OZ_G = 31.1035      # grams in one troy ounce

def block_tonnes(length_m, width_m, height_m, density_t_m3):
    """Tonnage of a rectangular block."""
    ...

def gold_ounces(tonnes, grade_gpt):
    ...

In [ ]:
grader.check("q1")

### Q2. Add `vshale`

`%%writefile -a` **appends** to the file. Add `vshale(gr, gr_clean=30.0, gr_shale=120.0)` that returns
$(GR - GR_{clean}) / (GR_{shale} - GR_{clean})$ clipped to 0–1 (your week 2 function, now with parameters
and default values), with a docstring that says what the parameters are.

In [ ]:
%%writefile -a geotools.py

def vshale(gr, gr_clean=30.0, gr_shale=120.0):
    ...

In [ ]:
grader.check("q2")

### Q3. Use the module

Reload `geotools`, read `block_model_pit1.csv` with `csv.DictReader`, and store in `total_t` the tonnes and in
`total_oz` the ounces of **all** blocks, using `geotools.block_tonnes` and `geotools.gold_ounces`.

In [ ]:
import csv, importlib
import geotools
importlib.reload(geotools)

with open("block_model_pit1.csv", encoding="utf-8") as f:
    blocks = list(csv.DictReader(f))

total_t = 0.0
total_oz = 0.0
...
total_t, total_oz

In [ ]:
grader.check("q3")

### Q4. The standard library does it for you

With the grades of the blocks (floats), store in `median_grade` and `stdev_grade` the median and the
sample standard deviation (module `statistics`), and in `rock_counts` a `Counter` of the `rock` column
(`from collections import Counter`). No loop needed.

In [ ]:
import statistics
from collections import Counter

grades = [float(b["au_gpt"]) for b in blocks]
median_grade = ...
stdev_grade = ...
rock_counts = ...
median_grade, stdev_grade, rock_counts

In [ ]:
grader.check("q4")

### Q5. A script that uses your module

Complete `report.py`: its `main()` reads the block model, uses `geotools` to compute the totals and writes
**one line** to `pit1_report.txt`, in this format:

```
Pit 1: 120 blocks, 123456 t, 1234.5 oz
```

(your numbers will differ: tonnes with 0 decimals, ounces with 1). Then run it with `%run report.py`.

In [ ]:
%%writefile report.py
"""Write the Pit 1 summary line to pit1_report.txt."""
import csv
import geotools

def main():
    ...

if __name__ == "__main__":
    main()

In [ ]:
%run report.py
print(open("pit1_report.txt", encoding="utf-8").read())

In [ ]:
grader.check("q5")

### Going further (not checked)

* Add `hydrostatic_mpa` and `distance_m` from weeks 3 and 5 to `geotools.py`, then `help(geotools)`.
* Download `geotools.py` and import it in your week 10 notebook: does `weighted_by_hole` get shorter?

---
**Before you leave:** run all the checks with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()